# W05-01 · TF-IDF와 BM25

필수 20분 + 확장 15분. 공식의 요소를 작은 예제로 계산한 뒤 실제 합성 자료에 적용합니다.

**진행 방식**: 먼저 실행 결과를 예측하고 셀을 실행합니다. 변경 실습은 제공된 값 하나를 바꿉니다. 핵심 셀은 네트워크·API 키 없이 실행됩니다. 모든 정책·질의는 교육용 합성 데이터입니다.

VS Code → 우측 상단 Select Kernel → 이 저장소의 .venv (Python 3.11). 노트북별로 Restart 후 Run All 가능합니다.

In [ ]:
from pathlib import Path
import sys, json, math, statistics
from dataclasses import asdict, replace
from collections import Counter
from time import perf_counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course_labs").is_dir()), None)
assert ROOT is not None, "저장소 내부에서 notebook을 열어 주세요."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_labs.retrieval import (
    Document,
    Chunk,
    LexicalIndex,
    load_documents,
    load_cases,
    chunk_document,
    tokenize,
    expand_query,
    eligible_chunks,
    reciprocal_rank_fusion,
    rerank_by_coverage,
    retrieval_metrics,
    retrieve,
    answer_from_evidence,
    evaluate,
)
from IPython.display import display, HTML

docs = load_documents()
cases = load_cases()
print("Python:", sys.version.split()[0], "| 문서:", len(docs), "| 평가 질의:", len(cases))
assert sys.version_info[:2] == (3, 11), "과정 기준 Python 3.11.x 커널 선택"

In [ ]:
def show(rows, columns=None):
    import html

    if not rows:
        return display(HTML("<p>결과 없음</p>"))
    cols = columns or list(rows[0])
    headers = "".join(
        '<th style="padding:8px;text-align:left">' + html.escape(str(c)) + "</th>" for c in cols
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:8px;border-bottom:1px solid #ddd">'
            + html.escape(str(r.get(c, "")))
            + "</td>"
            for c in cols
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            '<table style="font-size:15px;border-collapse:collapse"><tr>'
            + headers
            + "</tr>"
            + body
            + "</table>"
        )
    )

## 01 토큰화

한국어 조사와 띄어쓰기는 검색 결과에 영향을 줍니다.

In [ ]:
text = "상품 반품은 수령 후 가능합니다"
print("단어만:", tokenize(text, False))
print("문자 n-gram 포함:", tokenize(text, True))

**관찰·변경 과제**

반품/반품은/반품할 세 표현의 공통 토큰을 확인하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 02 문서 빈도

많은 문서에 등장하는 단어는 구분력이 낮을 수 있습니다.

In [ ]:
chunks = [c for d in docs for c in chunk_document(d)]
eligible = eligible_chunks(chunks, "2026-09-29")
index = LexicalIndex(eligible)
show(
    [
        {"term": t, "document_frequency": index.df[t]}
        for t in ["상품", "반품", "환불", "티켓", "포인트"]
    ]
)

**관찰·변경 과제**

여기서 document_frequency는 문서 조각 수 기준임을 기록하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 03 TF-IDF 계산

빈도와 역문서 빈도를 곱한 어휘 벡터입니다.

In [ ]:
N = 10
df = 2
tf = 3
idf = math.log((N + 1) / (df + 1)) + 1
weight = (1 + math.log(tf)) * idf
print("idf", round(idf, 4), "weight", round(weight, 4))

**관찰·변경 과제**

df=8로 바꾸고 구분력이 어떻게 달라지는지 설명하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 04 코사인 유사도

방향의 유사도를 비교하며 정답 확률로 해석하지 않습니다.

In [ ]:
def cosine(a, b):
    denominator = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(x * x for x in b))
    return sum(x * y for x, y in zip(a, b)) / denominator if denominator else 0


print(cosine([1, 2, 0], [2, 4, 0]), cosine([1, 0, 0], [0, 1, 0]))

**관찰·변경 과제**

벡터 길이만 두 배가 되어도 방향은 같은 이유를 설명하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 05 BM25 빈도 포화

같은 단어를 반복해도 점수 증가 폭이 줄어듭니다.

In [ ]:
k1 = 1.5
show([{"frequency": f, "saturation": round(f * (k1 + 1) / (f + k1), 4)} for f in [1, 2, 4, 8, 16]])

**관찰·변경 과제**

k1=0.5와 3.0을 비교하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 06 길이 정규화

문서가 길면 단어 등장 기회가 더 많습니다.

In [ ]:
avgdl = 100
f = 2
k1 = 1.5
show(
    [
        {
            "length": dl,
            "b": b,
            "tf_part": round(f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl)), 4),
        }
        for b in [0, 0.75, 1]
        for dl in [50, 100, 200]
    ]
)

**관찰·변경 과제**

b=0에서 길이 효과가 사라지는지 확인하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 07 동일 질의 비교

두 방식 모두 어휘 기반입니다. dense vs sparse 비교가 아닙니다.

In [ ]:
q = "반품 배송비와 환불 처리 기간"
for method in ["tfidf", "bm25"]:
    print(
        method,
        [(h["chunk"]["doc_id"], round(h["score"], 3)) for h in retrieve(q, method=method)["hits"]],
    )

**관찰·변경 과제**

서로 다른 방식의 raw score를 직접 더하면 왜 위험한가요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 08 토큰화 비교

형태소 분석기 없이도 n-gram 변화는 직접 실험할 수 있습니다.

In [ ]:
for ngrams in [False, True]:
    e = evaluate([c for c in cases if c["split"] == "dev"], ngrams=ngrams, k=3)
    print("ngrams:", ngrams, e["summary"])

**관찰·변경 과제**

짧은 공통 조각으로 무관한 결과가 늘어난 실패 질의를 찾아보세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 09 자기 질의

업무 용어와 사용자 표현을 분리합니다.

In [ ]:
my_queries = ["상품 되돌려 보내는 기간", "카드 취소 언제", "사이즈 갈아 주세요"]
for q in my_queries:
    print(q, [h["chunk"]["doc_id"] for h in retrieve(q)["hits"]])

**관찰·변경 과제**

사용자 인터뷰에서 실제 표현을 수집할 때 동의와 개인정보 최소화를 고려하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 10 어휘 검색의 한계

의미 유사도·최신성·권한은 별도의 문제입니다.

In [ ]:
print("TF-IDF/BM25: 어휘 일치의 순위")
print("권한/날짜: 검색 전 필터")
print("의미 검색: 학습된 임베딩 모델, 선택 W04-04")

**관찰·변경 과제**

본인 프로젝트에서 어휘 검색으로 충분한 경우와 의미 검색이 필요한 경우를 각각 쓰세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 완료 체크

- 설정 하나를 바꾼 비교 결과와 실패 질의 1개
- 출처·문서 버전·기준 날짜를 포함한 설명
- 측정한 것과 측정하지 않은 것의 구분

## 참고

- [RAG 원 논문](https://arxiv.org/abs/2005.11401)
- [Stanford IR: TF-IDF](https://nlp.stanford.edu/IR-book/html/htmledition/term-frequency-and-weighting-1.html)
- [Stanford IR: BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Ollama embed API](https://docs.ollama.com/api/embed)

로컬 구현 위치: course_labs/retrieval.py. 형태소 분석·학습된 임베딩·신경망 재정렬을 구현한 것으로 해석하지 않습니다.